In [1]:
from collections import Counter
from typing import TypeAlias

Bigram: TypeAlias = tuple[int, int]

In [2]:
def compute_bigram_statistics(
    token_ids: list[int], counter: Counter | None = None
) -> Counter[Bigram, int]:
    bigram_counter = Counter() if counter is None else counter
    for left, right in zip(token_ids, token_ids[1:]):
        bigram_counter[(left, right)] += 1
    return bigram_counter

def replace_bigram(token_ids: list[int], bigram: Bigram, bigram_id: int) -> list[int]:
    idx = 0
    new_token_ids = []
    while idx < len(token_ids):
        if token_ids[idx : idx + 2] == list(bigram):
            new_token_ids.append(bigram_id)
            idx += 2
        else:
            new_token_ids.append(token_ids[idx])
            idx += 1
    return new_token_ids

In [3]:
def merge(token_ids, vocab, merge_rules):
    # get bigram frequencies
    bigram_frequencies = compute_bigram_statistics(token_ids)

    # find the most frequent bigram lexicographically
    max_val = max(bigram_frequencies.values())
    filt_bigram_frequencies = {k: v for k, v in bigram_frequencies.items() if v == max_val}
    if len(filt_bigram_frequencies) > 1:
        filt_bigram_frequencies = dict(sorted(filt_bigram_frequencies.items()))
    merged_bigram = list(filt_bigram_frequencies.keys())[0]

    # merge the tokens and update the vocab and merge rules
    new_token = vocab[merged_bigram[0]] + vocab[merged_bigram[1]]
    merged_bigram_id = len(vocab)
    vocab = vocab + [new_token]
    merge_rules[merged_bigram] = merged_bigram_id

    new_token_ids = replace_bigram(token_ids, merged_bigram, merged_bigram_id)
    return new_token_ids, vocab, merge_rules, bigram_frequencies

In [4]:
s = 'aa acababa'
print(f"{len(s)}, {s}")

token_ids = list(s.encode("ascii"))
print(f"{len(token_ids)}, {token_ids}")

vocab = [chr(i) for i in range(128)]
print(f"{len(vocab)}")

merge_rules = {}

10, aa acababa
10, [97, 97, 32, 97, 99, 97, 98, 97, 98, 97]
128


In [5]:
token_id_dict = {}
freqs = {}
i = 0
while i < 5:
    token_id_dict[i] = token_ids
    new_token_ids, new_vocab, new_merge_rules, bigram_frequencies = merge(token_ids, vocab, merge_rules)
    token_ids, vocab, merge_rules = new_token_ids, new_vocab, new_merge_rules
    freqs[i] = bigram_frequencies
    i += 1
    print(f"{i+1}: {len(token_ids)}")
token_id_dict[i] = token_ids

2: 8
3: 7
4: 6
5: 5
6: 4


In [8]:
print([vocab[t] for t in token_ids])

['aa', ' a', 'cab', 'aba']


In [15]:
n_s = 'abacdaba'

n_token_ids = list(n_s.encode("ascii"))
print(f"{len(n_token_ids)}, {n_token_ids}")

8, [97, 98, 97, 99, 100, 97, 98, 97]


In [ ]:
print(n_token_ids)
for bigram, merged_id in merge_rules.items():
    n_token_ids = replace_bigram(n_token_ids, bigram, merged_id)
    print(n_token_ids)

[97, 98, 97, 99, 100, 97, 98, 97]
[128, 97, 99, 100, 128, 97]
[128, 97, 99, 100, 128, 97]
[128, 97, 99, 100, 128, 97]
[128, 97, 99, 100, 128, 97]
[132, 99, 100, 132]


In [18]:
print([vocab[t] for t in n_token_ids])

['aba', 'c', 'd', 'aba']


In [19]:
vocab[127:]

['\x7f', 'ab', ' a', 'aa', 'cab', 'aba']